In [ ]:
import os

# Install java
! apt-get update -qq
! apt-get install -y openjdk-8-jdk-headless -qq > /dev/null

os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-8-openjdk-amd64"
os.environ["PATH"] = os.environ["JAVA_HOME"] + "/bin:" + os.environ["PATH"]
! java -version

# Install pyspark
! pip install --ignore-installed -q pyspark==2.4.4
! pip install --ignore-installed -q spark-nlp==2.7.1

openjdk version "1.8.0_342"
OpenJDK Runtime Environment (build 1.8.0_342-8u342-b07-0ubuntu1~18.04-b07)
OpenJDK 64-Bit Server VM (build 25.342-b07, mixed mode)
     |████████████████████████████████| 215.7 MB 61 kB/s 
     |████████████████████████████████| 197 kB 66.3 MB/s 
     |████████████████████████████████| 138 kB 5.0 MB/s 


In [ ]:
import sparknlp

spark = sparknlp.start(gpu = True) # for GPU training >> sparknlp.start(gpu = True) # for Spark 2.3 =>> sparknlp.start(spark23 = True)

from sparknlp.base import *
from sparknlp.annotator import *
from pyspark.ml import Pipeline
import pandas as pd

print("Spark NLP version", sparknlp.version())

print("Apache Spark version:", spark.version)

spark

Spark NLP version 2.7.1
Apache Spark version: 2.4.4


In [ ]:
!gdown --id 10I-rrFBMKY7fm2hWDkQ6OjNS2B8uuQhD #new_labelled data
!unzip '/content/new_basic_human.zip'
!gdown --id 1rptRiR7bmrQYdk6P9a_wsTg6NvsNhur3
!unzip '/content/labelled_data.zip'


Streaming output truncated to the last 5000 lines.
  inflating: new_basic_human/1306@politics.txt  
  inflating: new_basic_human/1307@currentaffairs.txt  
  inflating: new_basic_human/1308@funtalks.txt  
  inflating: new_basic_human/1309@funtalks.txt  
  inflating: new_basic_human/130@others.txt  
  inflating: new_basic_human/1310@others.txt  
  inflating: new_basic_human/1311@currentaffairs.txt  
  inflating: new_basic_human/1312@funtalks.txt  
  inflating: new_basic_human/1313@currentaffairs.txt  
  inflating: new_basic_human/1314@funtalks.txt  
  inflating: new_basic_human/1315@currentaffairs.txt  
  inflating: new_basic_human/1316@funtalks.txt  
  inflating: new_basic_human/1317@currentaffairs.txt  
  inflating: new_basic_human/1318@funtalks.txt  
  inflating: new_basic_human/1319@currentaffairs.txt  
  inflating: new_basic_human/131@politics.txt  
  inflating: new_basic_human/1320@funtalks.txt  
  inflating: new_basic_human/1321@currentaffairs.txt  
  inflating: new_basic_human/13

In [ ]:
!gdown --id 11LXff9reIVVASkJe8CDm1pyAmuvKOTYi #new_data
!unzip '/content/new_data.zip'


In [ ]:
import os
from collections import Counter
import numpy as np
import pandas as pd

def make_datasetknnsemi():
    direc = "/content/labelled_data/"
    files = os.listdir(direc)
    emails = [direc + email for email in files]
    feature_set = []
    labels = []
    c = len(emails)
    words = []

    for email in emails:
        f = open(email ,encoding="utf8")
        words.append(f.read())
        if "politics" in email:
            labels.append('Politics')
        if "fun" in email:
            labels.append("Fun talk")
        if "tech" in email:
            labels.append("Technology")
        if "curr" in email:
            labels.append("Current Events")
        if "enter" in email:
            labels.append("Entertainment")
        if "other" in email:
            labels.append("Other")
    return words, labels

a,b = make_datasetknnsemi()
A = np.c_[a,b]
print(A)

[['11.Assets Recovery unit establish12. pM complaint cell establish in PM House. 13. Health card upto 5lac for fata, federa?'
  'Other']
 ['The Man at work  ' 'Technology']
 ['Still wondering why no one has come up with names of Bhatt Camp. Mahesh Bhatt And Vikram Bhatt are known for their? '
  'Current Events']
 ...
 [' Some Beautiful Creations By ' 'Other']
 ['Today is . As we are starting Wine & Whine in Nigeria we need to address the abysmal state of education wi?'
  'Current Events']
 ['shahi hamam  maybe  travel figure  beloved starting point' 'Other']]


In [ ]:
from pyspark.sql.functions import col
B = pd.DataFrame(A, columns=['Text','Type'])
df = spark.createDataFrame(B)
df.show(15)
df.groupBy("Type") \
    .count() \
    .orderBy(col('count').desc()) \
    .show()

+--------------------+--------------+
|                Text|          Type|
+--------------------+--------------+
|11.Assets Recover...|         Other|
|   The Man at work  |    Technology|
|Still wondering w...|Current Events|
|RAW and CIA Funde...|         Other|
|It's Official!Ste...|         Other|
|The latest The ad...|Current Events|
|Behold brothers a...|         Other|
|COAS General Qama...|         Other|
|Good to see COAS ...| Entertainment|
|Love is not to be...|Current Events|
|Another century f...|         Other|
|Pakistan is final...|Current Events|
|BOOM BOOM Leaving...|      Politics|
| . Hey... look at...|      Fun talk|
|Finally the big n...|Current Events|
+--------------------+--------------+
only showing top 15 rows

+--------------+-----+
|          Type|count|
+--------------+-----+
|Current Events|  856|
|         Other|  854|
|      Fun talk|  245|
|      Politics|  231|
|    Technology|   74|
| Entertainment|   73|
+--------------+-----+



In [ ]:
(train_df, val_df) = df.randomSplit([0.7, 0.3], seed = 8)
print("Training Dataset Count: " + str(train_df.count()))
print("Validation Dataset Count: " + str(val_df.count()))

Training Dataset Count: 1653
Validation Dataset Count: 680


In [ ]:
document_assembler = DocumentAssembler() \
.setInputCol("Text") \
.setOutputCol("document") \
.setCleanupMode("shrink")

tokenizer = Tokenizer() \
.setInputCols(["document"]) \
.setOutputCol("token") \
.setSplitChars(['-']) \
.setContextChars(['(', ')', '?', '!', '#', '@'])

normalizer = Normalizer() \
.setInputCols(["token"]) \
.setOutputCol("normalized")\
.setCleanupPatterns(["[^\w\d\s]"])

stopwords_cleaner = StopWordsCleaner()\
.setInputCols("normalized")\
.setOutputCol("cleanTokens")\
.setCaseSensitive(False)

lemma = LemmatizerModel.pretrained('lemma_antbnc') \
.setInputCols(["cleanTokens"]) \
.setOutputCol("lemma")

bert_embeddings = BertEmbeddings().pretrained(name='bert_base_cased', lang='en') \
.setInputCols(["document",'token'])\
.setOutputCol("embeddings")

embeddingsSentence = SentenceEmbeddings() \
.setInputCols(["document", "embeddings"]) \
.setOutputCol("sentence_embeddings") \
.setPoolingStrategy("AVERAGE")

classsifierdl = ClassifierDLApproach()\
.setInputCols(["sentence_embeddings"])\
.setOutputCol("class")\
.setLabelColumn("Type")\
.setMaxEpochs(10)\
.setLr(0.01)\
.setBatchSize(8) \
.setEnableOutputLogs(True)
#.setOutputLogsPath('logs')

pipeline = Pipeline(
    stages=[document_assembler,
            tokenizer,
            normalizer,
            stopwords_cleaner,
            lemma,
            bert_embeddings,
            embeddingsSentence,
            classsifierdl])

lemma_antbnc download started this may take some time.
Approximate size to download 907.6 KB
[OK!]
bert_base_cased download started this may take some time.
Approximate size to download 389.1 MB
[OK!]


In [ ]:
%%time
pipelineModel = pipeline.fit(train_df)

CPU times: user 1.51 s, sys: 172 ms, total: 1.68 s
Wall time: 3min 41s


In [ ]:
preds = pipelineModel.transform(val_df)

In [ ]:
preds.select('Text','Type',"class.result").show(10, truncate=100)

+----------------------------------------------------------------------------------------+--------------+-------+
|                                                                                    Text|          Type| result|
+----------------------------------------------------------------------------------------+--------------+-------+
|                                                  i'm curious, These Generals lost eve? |         Other|[Other]|
|                        Ambassador of the EU to Pakistan His Excellency Jean-Francois C?|         Other|[Other]|
|                       And I'm tired of all # things. -I am a man and I have been sexu? |      Politics|[Other]|
|                                                           "Mother of all Terrorism"   ?|Current Events|[Other]|
|                                                  , have been undermined, says READ?? h?|Current Events|[Other]|
| . Hey... look at that! That hashtag really DOES have a use other than over-rated, sel?

In [ ]:
preds_df = preds.select('Text','Type',"class.result").toPandas()

# The result is an array since in Spark NLP you can have multiple sentences.
# Let's explode the array and get the item(s) inside of result column out
preds_df['result'] = preds_df['result'].apply(lambda x : x[0])

In [ ]:
# We are going to use sklearn to evalute the results on test dataset
from sklearn.metrics import classification_report

print(classification_report(preds_df['result'], preds_df['Type']))

                precision    recall  f1-score   support

Current Events       0.00      0.00      0.00         0
 Entertainment       0.00      0.00      0.00         0
      Fun talk       0.00      0.00      0.00         0
         Other       1.00      0.34      0.51       680
      Politics       0.00      0.00      0.00         0
    Technology       0.00      0.00      0.00         0

      accuracy                           0.34       680
     macro avg       0.17      0.06      0.09       680
  weighted avg       1.00      0.34      0.51       680



/usr/local/lib/python3.7/dist-packages/sklearn/metrics/_classification.py:1318: UndefinedMetricWarning: Recall and F-score are ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/usr/local/lib/python3.7/dist-packages/sklearn/metrics/_classification.py:1318: UndefinedMetricWarning: Recall and F-score are ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/usr/local/lib/python3.7/dist-packages/sklearn/metrics/_classification.py:1318: UndefinedMetricWarning: Recall and F-score are ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
